# SLC — YOLOv8 tile detector training

**Before running:** `Runtime → Change runtime type → T4 GPU`

Steps:
1. Mount Drive (saves checkpoints — Colab disconnects after ~12h)
2. Clone repo + install deps
3. Generate synthetic dataset
4. Train 20 epochs on GPU (~12 min)
5. Export to ONNX
6. Copy model to Drive + back to repo

In [ ]:
# ── Cell 1: Mount Drive first — checkpoints saved here ──────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/slc_runs'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted. Checkpoints will go to:', DRIVE_DIR)

In [ ]:
# ── Cell 2: GPU check ────────────────────────────────────────────────────────
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# ── Cell 3: Clone repo + install deps ───────────────────────────────────────
import os

REPO = 'https://github.com/IL-RY-byte/slc.git'

if not os.path.exists('/content/slc'):
    !git clone {REPO} /content/slc
else:
    !cd /content/slc && git pull

%cd /content/slc

!pip install -q ultralytics reedsolo pillow
print('deps installed')

In [ ]:
# ── Cell 4: Generate synthetic dataset ──────────────────────────────────────
# 6000 images takes ~4 min on Colab CPU
# If you re-run the notebook, skip this cell if dataset/ already exists
import os

if not os.path.exists('dataset/images/train'):
    !python scripts/generate_yolo_data.py --out dataset/ --n 6000
else:
    # Count existing images
    n = len(list(__import__('pathlib').Path('dataset/images/train').glob('*.jpg')))
    print(f'Dataset already exists: {n} train images — skipping generation')

In [ ]:
# ── Cell 5: Train ────────────────────────────────────────────────────────────
from ultralytics import YOLO

DRIVE_DIR = '/content/drive/MyDrive/slc_runs'

model = YOLO('yolov8n-seg.pt')  # nano-seg, ~3.3M params

results = model.train(
    data='dataset/tile_detect.yaml',
    epochs=20,
    imgsz=320,
    batch=64,          # fits in T4 16 GB easily
    device=0,          # GPU
    workers=4,
    # Save to Drive so checkpoints survive disconnection
    project=DRIVE_DIR,
    name='train',
    exist_ok=True,     # resume into same folder
    save_period=5,     # checkpoint every 5 epochs
    cache='ram',       # load all images into RAM once — much faster
    # Augmentations
    fliplr=0.5,
    mosaic=0.5,
    mixup=0.1,
    degrees=0.0,       # tiles have no top/bottom convention
)

print()
print('mAP50 :', results.results_dict.get('metrics/mAP50(B)', 'n/a'))
print('Best  :', results.save_dir)

In [ ]:
# ── Cell 6: Export to ONNX ───────────────────────────────────────────────────
from pathlib import Path
from ultralytics import YOLO
import shutil

DRIVE_DIR = '/content/drive/MyDrive/slc_runs'
best_pt  = Path(DRIVE_DIR) / 'train' / 'weights' / 'best.pt'

if not best_pt.exists():
    raise FileNotFoundError(f'best.pt not found at {best_pt}. Did training finish?')

print('Exporting', best_pt)
m = YOLO(str(best_pt))
onnx_path = m.export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False)
onnx_path = Path(onnx_path)

# Copy to Drive for easy download
dest_drive = Path(DRIVE_DIR) / 'tile_detector.onnx'
shutil.copy2(onnx_path, dest_drive)

# Also copy to repo assets folder
dest_repo = Path('/content/slc/flutter/assets/models/tile_detector.onnx')
shutil.copy2(onnx_path, dest_repo)

size_mb = dest_drive.stat().st_size / 1024 / 1024
print(f'Saved to Drive: {dest_drive}  ({size_mb:.1f} MB)')
print(f'Saved to repo : {dest_repo}')

In [ ]:
# ── Cell 7: Push model back to GitHub ────────────────────────────────────────
# Set your GitHub token once (Runtime → Secrets, key=GH_TOKEN) or paste here.
import os
from google.colab import userdata

try:
    token = userdata.get('GH_TOKEN')
except Exception:
    token = ''  # push will fail if repo is private without token

REPO_URL = f'https://{token}@github.com/IL-RY-byte/slc.git' if token else 'https://github.com/IL-RY-byte/slc.git'

!cd /content/slc && git config user.email 'colab@slc' && git config user.name 'Colab'

# Uncomment tile_detector.onnx asset in pubspec.yaml
!cd /content/slc && sed -i 's|^    #.*tile_detector.onnx.*|    - assets/models/tile_detector.onnx      # YOLOv8-nano-seg tile detector|' flutter/pubspec.yaml

!cd /content/slc && git add flutter/assets/models/tile_detector.onnx flutter/pubspec.yaml
!cd /content/slc && git commit -m 'Add trained tile_detector.onnx (YOLOv8-nano-seg, 20 epochs)'
!cd /content/slc && git push {REPO_URL} main

print('Done! Rebuild Flutter web after pulling this commit.')

## After the notebook finishes

On your Windows machine:

```bash
cd slc
git pull

# uncomment in pubspec.yaml (already done by Cell 7):
#   - assets/models/tile_detector.onnx

cd flutter
flutter build web --release --pwa-strategy=none
copy web\ort_bridge.js build\web\ort_bridge.js

# restart server
cd build\web
python -m http.server 9090
```

The app will now use the real YOLOv8 detector instead of Hough circles.
Check the header — it should say `web · yolo + onnx`.

## If Colab disconnects mid-training

Training is saved to Drive every 5 epochs. To resume:
```python
from ultralytics import YOLO
model = YOLO('/content/drive/MyDrive/slc_runs/train/weights/last.pt')
model.train(resume=True)
```